# FP label audit: Puma (#182)

Triage of the Puma-brand rows that enter the FP classifier's training set, for label errors in **both directions**:

- a positive (`labeled` / `skipped`) that is about another meaning of "Puma" (the animal, the Ford Puma car, Puma Energy);
- a negative (`false_positive`) that is about Puma the company: its products, deals or legal disputes.

The second direction is the more harmful one. It teaches the classifier to skip real Puma coverage, and the classifier's own skips then feed later training sets (#154).

**This notebook is read-only.** It reads the database and the served FP API and writes nothing. It produces a **candidate list for human adjudication**. Corrections are applied separately, by the human's ruling, through `scripts/fix_label.py` or a targeted relabel (see #182).

**Re-running:** needs PostgreSQL (`docker compose up -d`) and the FP API on `localhost:8000` (`docker compose up fp-classifier-api`). Outputs are a snapshot of the database at run time. Re-run the notebook rather than quoting its numbers elsewhere.

Related: #154 (label provenance; its PR2 retrain consumes these corrections), #155 (review of classifier-only negatives), #187 (the same audit for other brand-name collisions).

## Setup

In [1]:
# Standard library
import json
import re
import sys
import urllib.request
from pathlib import Path

# Third-party
import pandas as pd
from sqlalchemy import text

# Project root on the path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Project modules
from src.data_collection.database import db

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 200)

BRAND = "Puma"
FP_API_URL = "http://localhost:8000"
TRAINING_STATUSES = ("labeled", "skipped", "false_positive")  # export_training_data.py: fp dataset
BATCH_SIZE = 50

# Other meanings of the brand name (positive-class red flags)
OTHER_SENSE = {
    "wildlife": re.compile(r"mountain lion|cougar|wildlife|big cat|\bzoo\b|park rangers?|predator|catamount|feline|penguins?", re.I),
    "vehicle": re.compile(r"\bFord\b|Gen[- ]?E\b|crossover|hatchback|\bSUV\b", re.I),
    "other company": re.compile(r"Puma Energy|Puma Biotech|Super Puma|helicopter", re.I),
    "person": re.compile(r"\bLa Puma\b|\bDr\.? Puma\b", re.I),
}
# docs/LABELING.md "Stock Article Classification": template stock/finance content is false_positive by policy
STOCK_TEMPLATE = re.compile(
    r"\((?:OTCMKTS|ETR|NYSE|NASDAQ)\s*:\s*\w+\s*\)|Short Interest|Moving Average|Consensus (?:Rating|Recommendation)"
    r"|Average (?:Rating|Recommendation)|Expected to Announce Quarterly Earnings|Projected to Post|Get Free Report"
    r"|most recent 13F filing|Stock Price (?:Up|Down)|Trading (?:Up|Down)", re.I)
# Brand-sense vocabulary (negative-class red flags)
BRAND_SENSE = re.compile(r"sneaker|footwear|trainers|apparel|sportswear|trademark|athleisure|\bshoes?\b|kit deal|sponsorship", re.I)

## Training rows with provenance

Provenance follows #154's design (D-9.1): the **latest** FP `classifier_predictions` row decides it. `skipped_llm` means the classifier labeled the article itself (`fp_classifier`). `continued_to_llm` or `failed` means the LLM did (`llm`). No FP row means the article predates the classifier (`pre_classifier`). The human overlay (#143 corrections, #146 confirmed samples) does not exist yet, so no row is `human`.

In [2]:
QUERY = text("""
    SELECT a.id, a.labeling_status, a.title, coalesce(a.full_content, '') AS full_content,
           a.brands_mentioned, a.source_name, a.category, a.created_at,
           p.action_taken AS fp_action
    FROM articles a
    LEFT JOIN LATERAL (
        SELECT action_taken FROM classifier_predictions p
        WHERE p.article_id = a.id AND p.classifier_type = 'fp'
        ORDER BY p.created_at DESC LIMIT 1
    ) p ON true
    WHERE :brand = ANY(a.brands_mentioned) AND a.labeling_status IN :statuses
""")

with db.get_session() as session:
    rows = session.execute(QUERY, {"brand": BRAND, "statuses": TRAINING_STATUSES}).mappings().all()
df = pd.DataFrame(rows)

def provenance(action):
    if action is None or pd.isna(action):
        return "pre_classifier"
    return "fp_classifier" if action == "skipped_llm" else "llm"

df["provenance"] = df["fp_action"].map(provenance)
df["label"] = (df["labeling_status"] != "false_positive").astype(int)  # 1 = sportswear
pd.crosstab(df["labeling_status"], df["provenance"], margins=True)

provenance,fp_classifier,llm,pre_classifier,All
labeling_status,,,,
false_positive,43,20,32,95
labeled,0,61,13,74
skipped,3,136,26,165
All,46,217,71,334


## Served model scores

Every row is scored by the FP API the labeling pipeline calls. The version it reports is recorded so the snapshot says which model produced the scores.

In [3]:
def api(path, payload=None):
    data = json.dumps(payload).encode() if payload is not None else None
    req = urllib.request.Request(FP_API_URL + path, data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=300) as resp:
        return json.load(resp)

model_info = api("/model/info")
registry = json.loads((project_root / "models" / "registry.json").read_text())
pointer = registry["fp"]["production"]
trained_at = pd.Timestamp(registry["fp"]["versions"][pointer]["created_at"]).tz_convert("UTC")
served = model_info.get("version")
print(f"served version: {served}  registry pointer: {pointer}  pointer created_at: {trained_at}")
if served is None:
    print("NOTE: the served API does not report a version (an image built before #115); the held-out cut uses the pointer's date.")
elif served != pointer:
    print("WARNING: the served version differs from the registry pointer; the held-out cut uses the pointer's date.")

probs, preds = [], []
for start in range(0, len(df), BATCH_SIZE):
    chunk = df.iloc[start:start + BATCH_SIZE]
    articles = [
        {"title": r.title or "", "content": r.full_content, "brands": list(r.brands_mentioned or []),
         "source_name": r.source_name, "category": list(r.category) if r.category else None}
        for r in chunk.itertuples()
    ]
    out = api("/predict/batch", {"articles": articles})
    for p in out["predictions"]:
        probs.append(p["probability"])
        preds.append(int(p["is_sportswear"]))
df["probability"] = probs
df["predicted"] = preds
df["disagrees"] = df["predicted"] != df["label"]
df["held_out"] = pd.to_datetime(df["created_at"], utc=True) > trained_at
pd.crosstab(df["label"], df["predicted"], rownames=["label"], colnames=["served model"])

served version: None  registry pointer: v2.4.0  pointer created_at: 2026-01-16 02:15:36+00:00
NOTE: the served API does not report a version (an image built before #115); the held-out cut uses the pointer's date.


served model,0,1
label,,
0,74,21
1,2,237


## Other-meaning and brand-meaning flags

In [4]:
head = df["title"].fillna("") + " " + df["full_content"].str[:3000]

def other_sense(text_):
    return ",".join(name for name, rx in OTHER_SENSE.items() if rx.search(text_)) or None

df["other_sense"] = head.map(other_sense)
df["brand_sense"] = head.str.contains(BRAND_SENSE)
df["flag"] = None
df.loc[df["disagrees"], "flag"] = "model disagrees"
df.loc[(df["label"] == 1) & df["other_sense"].notna(), "flag"] = "positive, other meaning"
df.loc[(df["label"] == 0) & df["brand_sense"], "flag"] = "negative, brand vocabulary"
df.loc[(df["label"] == 0) & df["brand_sense"] & df["disagrees"], "flag"] = "negative, brand vocabulary + model disagrees"
df["stock_template"] = df["title"].fillna("").str.contains(STOCK_TEMPLATE) | df["full_content"].str[:3000].str.contains(STOCK_TEMPLATE)
df.loc[(df["label"] == 1) & df["stock_template"], "flag"] = "positive, stock template"

# Negatives that are correct by policy are set aside, not dropped: shown below so the human can skim them.
df["set_aside"] = None
neg = df["label"] == 0
df.loc[neg & df["flag"].notna() & df["stock_template"], "set_aside"] = "stock template (false_positive by policy)"
# brand vocabulary overrides an other-meaning match: the regexes are noisy and a brand article must stay a candidate
df.loc[neg & df["flag"].notna() & df["set_aside"].isna() & df["other_sense"].notna() & ~df["brand_sense"], "set_aside"] = "other meaning of the name"
pd.crosstab(df["flag"].fillna("none"), df["set_aside"].fillna("candidate"))

set_aside,candidate,other meaning of the name,stock template (false_positive by policy)
flag,,,
model disagrees,2,5,1
"negative, brand vocabulary",1,0,23
"negative, brand vocabulary + model disagrees",4,0,10
none,264,0,0
"positive, other meaning",6,0,0
"positive, stock template",18,0,0


## Candidates for human adjudication

A candidate is a row the served model disagrees with, a positive carrying other-meaning or stock-template vocabulary, or a negative carrying brand vocabulary. **Negatives that are `false_positive` by the labeling policy are set aside**: template stock articles (`docs/LABELING.md`, "Stock Article Classification") and articles about another meaning of the name. They are listed after the candidates, not dropped. **A flag is a reason to look, not a verdict.** The human rules each one. Rows are ordered so the likeliest errors come first: negatives the model scores highest, then positives it scores lowest.

In [5]:
COLUMNS = ["id", "labeling_status", "provenance", "probability", "held_out", "other_sense", "flag", "title"]
candidates = df[df["flag"].notna() & df["set_aside"].isna()].copy()
candidates["_order"] = candidates.apply(lambda r: -r.probability if r.label == 0 else r.probability - 1, axis=1)
candidates = candidates.sort_values(["label", "_order"])[COLUMNS]
print(f"{len(candidates)} candidates")
candidates

31 candidates


,id,labeling_status,provenance,probability,held_out,other_sense,flag,title
101,3a637c91-2fd3-4f8e-86be-f1ea45780808,false_positive,llm,0.934687,True,None,"negative, brand vocabulary + model disagrees",CLOT and adidas Celebrate the Year of the Horse in This Week's Best Footwear Drops
81,dcf616c4-7fb5-47b2-b430-6175902ba489,false_positive,llm,0.793489,True,person,"negative, brand vocabulary + model disagrees",Joe La Puma talks career evolution and consistency
34,8dca1142-c86c-4f26-a0e8-d031ffbf65df,false_positive,pre_classifier,0.708599,False,None,"negative, brand vocabulary + model disagrees",UK Menswear Market Outlook 2035: Sustainable Fashion and Lifestyle Shifts Drive a Robu...
332,f751891e-1ed4-4183-a907-5d198022f8e9,false_positive,fp_classifier,0.507079,True,None,"negative, brand vocabulary + model disagrees",Puma Warehouse Inside Job Exposed: Stock-in Charges Arrested; 43% rise in Cognisable C...
233,1739fb42-ee49-4466-aeec-c39894d2ba41,false_positive,fp_classifier,0.503945,True,None,model disagrees,Style (04/13/26)
169,d79efa45-c131-44ff-a233-d0b4f7c97858,false_positive,fp_classifier,0.407576,True,wildlife,"negative, brand vocabulary",Puma trademark challenge against Tiger Wood Sun Day Red fails
134,aabc7f6e-8fc5-4c02-9880-b9e404bd840a,skipped,fp_classifier,0.296382,False,None,"positive, stock template",Puma Se (OTCMKTS:PUMSY) Sees Large Drop in Short Interest
187,55b54027-afc5-4566-ab7f-f6d393bc8343,labeled,llm,0.473515,False,None,model disagrees,"Siemens , Puma , Nuremberg airport : How much Christmas bonus will companies pay in Fr..."
188,f69140d1-cf99-48a2-8663-22dbae05af1a,skipped,llm,0.541230,True,None,"positive, stock template",Short Interest in Puma Se (OTCMKTS:PUMSY) Drops By 75.2%
280,16497d46-e8cc-49b6-bd2d-2f0c297d5f5a,skipped,llm,0.571382,True,None,"positive, stock template",Puma Se (OTCMKTS:PUMSY) Short Interest Up ? in July


In [6]:
set_aside = df[df["set_aside"].notna()].sort_values(["set_aside", "probability"], ascending=[True, False])
set_aside[["id", "provenance", "probability", "set_aside", "title"]]

,id,provenance,probability,set_aside,title
181,4139a07f-e0e0-488a-bc6d-7ef3a893a668,llm,0.907552,other meaning of the name,APO Group - Africa Newsroom / Press release | Puma Energy and Hass Petroleum Group ann...
203,2109af68-605e-4e83-a9ca-9bfa1dae52cf,llm,0.841029,other meaning of the name,"INTERVIEW : Dr . Puma Talks Sorin Heart Scan , Power 105 . 1 Partnership , U . S . | K97"
182,7ddaa60a-9412-43fc-9424-d1ab5294534d,llm,0.779610,other meaning of the name,Puma Energy and Hass Petroleum Group announce lubricant distribution partnership
173,8ff1091c-7223-4e7b-8aeb-57ffdb4a57e4,llm,0.641020,other meaning of the name,NAF commissions reactivated Super Puma at Ikeja Base
48,7cb9e80f-91ce-4556-84a5-8563b9cb0c21,pre_classifier,0.574601,other meaning of the name,Leonardo warns Yeovil helicopter factory at risk without Puma replacement deal
190,8e82c33b-de3b-4369-8504-c31fc1be5c9b,llm,0.796433,stock template (false_positive by policy),Puma Se (OTCMKTS:PUMSY) Given Average Rating of “Hold” by Brokerages
89,49a643c4-248c-4ff1-8669-e751c4f7c885,llm,0.787784,stock template (false_positive by policy),Puma Se (OTCMKTS:PUMSY) Receives Consensus Rating of “Hold” from Brokerages
156,647d10ef-5880-4466-b8e2-f24beff5dda4,llm,0.753710,stock template (false_positive by policy),"Universal Beteiligungs und Servicegesellschaft mbH Buys 159 , 200 Shares of Puma Biote..."
186,76716a40-4eb8-4ed0-8774-44d3bf9dc64c,llm,0.671452,stock template (false_positive by policy),Puma (PMMAF) Expected to Announce Quarterly Earnings on Thursday
191,c426088d-efc0-412b-933f-68c2769732fa,llm,0.663211,stock template (false_positive by policy),Puma Se (OTCMKTS:PUMSY) Receives Consensus Recommendation of “Hold” from Brokerages


## AC3: are real wildlife-Puma negatives under-represented?

Measuring this by the served model's scores alone would be circular: a `fp_classifier` negative carries a label the model wrote itself. So the primary instrument is **provenance**: how many wildlife-Puma negatives survive #154's default exclusion (D-9.3: exclude an FP negative whose latest FP decision is `skipped_llm`, unless a human verified it; no human overlay exists yet). The served-model score on held-out rows, collected after the production model's training date, is secondary evidence.

**Criterion (instrument named, no fixed threshold):** wildlife-Puma negatives are under-represented if few or none survive the exclusion. If so, real examples are collected and labeled through the normal path (a follow-up the human approves). Synthetic text never goes into the training set.

In [7]:
# A wildlife match that also carries brand vocabulary or a vehicle match is not a wildlife article
is_wildlife = df["other_sense"].fillna("").str.contains("wildlife") & ~df["brand_sense"] & ~df["other_sense"].fillna("").str.contains("vehicle")
wildlife_neg = df[(df["label"] == 0) & is_wildlife].copy()
wildlife_neg["survives_154_exclusion"] = wildlife_neg["provenance"] != "fp_classifier"
display(pd.crosstab(wildlife_neg["provenance"], wildlife_neg["survives_154_exclusion"], margins=True))
wildlife_neg[["id", "provenance", "survives_154_exclusion", "held_out", "probability", "title"]].sort_values("probability", ascending=False)

survives_154_exclusion,False,True,All
provenance,,,
fp_classifier,3,0,3
pre_classifier,0,6,6
All,3,6,9


,id,provenance,survives_154_exclusion,held_out,probability,title
165,7dee38ea-74fc-4727-9a16-6c46da185b58,fp_classifier,False,True,0.273684,Santa Cruz Mountains | Puma protections could become permanent
301,a2000abd-0d61-40b0-ad55-738ce57982e5,fp_classifier,False,True,0.200000,PUMA FEST and the community Mongabay Latam has helped build
30,40670bcf-9b00-4f65-94fe-027146f298c0,pre_classifier,True,False,0.174242,Caldwell Zoo announces death of male puma
161,15415d1f-3dec-4e8d-b3a7-57802acc8519,fp_classifier,False,True,0.100000,Puma - like big cat sighting reignites mystery of legendary beast roaming united ki...
28,d9c81aaf-0c47-4ca4-a595-f00db24542eb,pre_classifier,True,False,0.057576,Caldwell Zoo beloved puma Monte dies at 16
37,c3d207a5-ebae-40c6-83d4-b30d4f211e58,pre_classifier,True,False,0.050000,Chile Puma Boom When Patagonia Ghost Cat Finally Shows Itself
10,b014dc47-ffae-4247-9674-bf11c552a60e,pre_classifier,True,False,0.033333,"Puma populations are on the rise in Patagonia , now targeting penguins as prey"
18,c1726d96-b28e-4cc5-a807-024a13605b81,pre_classifier,True,False,0.033333,How a new diet of penguins is changing puma behavior and social lives in Patagonia
32,9b937527-216c-47a0-97ed-199b0dca26e9,pre_classifier,True,False,0.008333,Patagonia puma population soars thanks to unexpected prey : penguins | Science


## Reading the results

- **Set aside:** negatives that the labeling policy makes `false_positive`. The served model scoring some of them as sportswear is a property of the model, not a label error. They are listed so the human can confirm the policy reading, not adjudicated row by row.
- **Candidates:** handed to the human for a per-row ruling. A confirmed brand-meaning negative is corrected to the positive class, and a confirmed other-meaning positive to `false_positive`. How each correction is applied, a relabel or a `fix_label.py` status update, is decided with the human in #182, because a targeted relabel still runs the FP pre-filter.
- **AC3:** read the provenance table against the criterion above. The held-out scores show whether the served model already gets real wildlife articles right. They cannot show whether the next training set will still contain them; that is the provenance table's job.
- Corrections are recorded on #182 (article IDs, old and new status), not in this notebook.